# WannaCry Incident Response Runbook

**Author**: Stan Toman

**Comprehensive Automated Incident Response Guide**

This runbook provides a complete, automated workflow for responding to WannaCry incidents from detection through post-incident activities.




**Version**: 0.1  
**Classification**: Internal Use Only  
**Severity**: CRITICAL

---


## 1. Executive Summary

This comprehensive incident response runbook provides step-by-step procedures for detecting, containing, eradicating, and recovering from WannaCry ransomware infections. The runbook is structured to follow the standard incident response lifecycle, integrating with the automated threat hunting system and including decision trees, response playbooks, and recovery procedures for each phase.

**Important Note:** This runbook is extensive and serves as a comprehensive template. Organizations must tailor this runbook to align with their specific systems, infrastructure, tools, and organizational Service Level Agreements (SLAs). Customization should include adjusting timelines, escalation procedures, contact information, technical commands for your specific environment, and response priorities based on your organizational risk profile and business requirements.

### 1.1. Quick Reference

| Attribute | Details |
| :--- | :--- |
| **Malware Name** | WannaCry, WCry, WanaCrypt0r, Wanna Decryptor |
| **Threat Level** | CRITICAL |
| **Primary Vector** | EternalBlue (MS17-010) SMB exploitation |
| **Propagation** | Self-propagating worm (lateral movement) |
| **Impact** | File encryption, system disruption, data loss |
| **Response Time** | Immediate isolation required upon confirmation |

---


## 2. Table of Contents

1. [Executive Summary](#summary)
2. [Table of Contents](#toc)
3. [Phase 1: Preparation & Prevention](#preparation)
4. [Phase 2: Detection & Analysis](#detection)
5. [Phase 3: Containment Procedures](#containment)
6. [Phase 4: Eradication Steps](#eradication)
7. [Phase 5: Recovery & Restoration](#recovery)
8. [Phase 6: Post-Incident Activities & Continuous Improvement](#post-incident)
9. [Appendix A: End-to-End Incident Checklist](#checklist)
10. [Appendix B: Communication Templates](#communication)
11. [Appendix C: Tools & Resources](#tools)
12. [Appendix D: Decision Trees](#decision-trees)
13. [Appendix E: Full Automation Scripts](#automation-scripts)

---


## 3. Phase 1: Preparation & Prevention {#preparation}

### 3.1. Description

Pre-incident preparation is the most critical phase for mitigating the impact of a potential WannaCry attack. This involves hardening systems, ensuring a robust backup strategy, and setting up comprehensive monitoring and detection capabilities. A well-prepared environment can prevent an infection altogether or significantly reduce its spread.

### 3.2. Procedures

#### 3.2.1. System Hardening

**Windows Systems:**
- Apply Microsoft Security Bulletin **MS17-010** patches immediately to all Windows systems.
- Disable the **SMBv1** protocol, which is highly vulnerable to EternalBlue exploitation.
- Enable the Windows Firewall with strict inbound rules, denying all traffic by default except for explicitly allowed services.
- Disable unnecessary services, especially RDP (Remote Desktop Protocol) if not in use, to reduce the attack surface.
- Implement application whitelisting with AppLocker or Windows Defender Application Control to prevent unauthorized executable execution.

**Network Level:**
- Block **SMB port 445 (TCP/UDP)** at the external firewall to prevent external attacks.
- Implement network segmentation to limit lateral movement, separating user workstations from servers and critical infrastructure.
- Deploy and configure Intrusion Detection/Prevention Systems (IDS/IPS) to monitor for exploit attempts.
- Actively monitor for EternalBlue exploitation attempts using network signatures and behavioral analysis.

#### 3.2.2. Backup Strategy

- Implement the **3-2-1 backup rule**: Maintain 3 copies of your data on 2 different media types, with 1 copy offsite.
- Ensure backups are disconnected from the network (air-gapped) or configured with immutability features to prevent ransomware from corrupting them.
- Periodically test backup restoration procedures to ensure they are effective and meet your Recovery Time Objective (RTO) and Recovery Point Objective (RPO).
- Document the backup strategy and ensure all stakeholders are aware of the procedures.

#### 3.2.3. Monitoring & Detection Setup

**Enable Logging:**
- Configure comprehensive logging for Windows Event Logs (Security, System, Application) on all systems.
- Deploy Sysmon (System Monitor) for advanced process monitoring and detection of suspicious behavior.
- Implement File Integrity Monitoring (FIM) to detect unauthorized changes to critical files.
- Configure network flow monitoring (NetFlow/sFlow) to capture and analyze network traffic patterns.

**Deploy Threat Hunting System:**
- Configure and enable the automated threat hunting system to perform automated IOC matching.
- Set up ML-based anomaly detection to identify unusual patterns of activity.
- Configure behavioral analysis rules to detect file encryption, mass file writes, and network scanning.

#### 3.2.4. Incident Response Team Setup

**Key Roles:**
- **Incident Commander (IC):** Oversees the entire response and coordinates all teams.
- **Technical Lead:** Manages technical aspects of the response.
- **Forensics Specialist:** Collects and analyzes forensic evidence.
- **Communications Officer:** Manages internal and external communications.
- **Executive Sponsor:** Provides executive support and decision-making authority.

| Role | Contact | Phone | Email |
| :--- | :--- | :--- | :--- |
| Incident Commander | [NAME] | [PHONE] | [EMAIL] |
| Technical Lead | [NAME] | [PHONE] | [EMAIL] |
| Forensics Specialist | [NAME] | [PHONE] | [EMAIL] |
| Communications Officer | [NAME] | [PHONE] | [EMAIL] |
| Executive Sponsor | [NAME] | [PHONE] | [EMAIL] |
| SOC Contact | [NAME] | [PHONE] | [EMAIL] |

---


## 4. Phase 2: Detection & Analysis {#detection}

### 4.1. Description

This phase focuses on identifying a WannaCry infection as quickly as possible. Detection can be achieved through Indicator of Compromise (IOC) matching, which looks for known malicious files and network traffic, and behavioral analysis, which identifies suspicious patterns of activity. Early detection is critical to minimizing the spread and impact of the infection.

### 4.2. Procedures

#### 4.2.1. IOC-Based Detection

**File Hashes:**
- MD5: `db349b97c37d22f5ea1d1841e3c89eb4` (dropper)
- SHA256: `24d004a104d4d54034dbcffc2a4b19a11f39008a575aa614ea04703480b1022c` (dropper)
- MD5: `84c82835a5d21bbcf75a61706d8ab549` (dropped executable)

**Network Indicators:**
- Killswitch Domain: `iuqerfsodp9ifjaposdfjhgosurijfaewrwergwea.com`
- Tor C2 Nodes: `gx7ekbenv2riucmf.onion`, `57g7spgrzlojinas.onion`
- Bitcoin Wallets: `13AM4VW2dhxYgXeQepoHkHSQuy6NgaEb94`

**Process & Service Indicators:**
- Executable: `tasksche.exe`, `qeriuwjhrf`, `@WanaDecryptor@.exe`
- Service Name: `mssecsvc2.0` (Microsoft Security Center 2.0 Service)

**Registry Indicators:**
- Path: `HKLM\SYSTEM\CurrentControlSet\Services\mssecsvc2.0`

**Mutex Objects:**
- `Global\WINDOWS_TASKOSHT_MUTEX`
- `Global\WINDOWS_TASKCST_MUTEX`
- `Global\MsWinZonesCacheCounterMutexA`

#### 4.2.2. Behavioral Detection

**File System Behavior:**
- Mass file write operations (>100 files/minute) indicating encryption activity.
- Files being renamed with the `.WNCRY` extension.
- Creation of ransom notes (`DECRYPT_INSTRUCTIONS.txt`, `@WanaDecryptor@.txt`).
- Modification of desktop wallpaper to display ransom message.

**Network Behavior:**
- Internal scanning on SMB port 445 (lateral movement attempts).
- Outbound connections to Tor exit nodes for C2 communication.
- Bitcoin wallet queries to check payment status.
- Unusual outbound connections to external IP addresses.

**Process Behavior:**
- Service installation (`mssecsvc2.0`).
- Process injection and DLL loading from suspicious locations.
- Elevated privilege operations by non-administrative users.
- Registry modifications to disable security software or create persistence mechanisms.

### 4.3. Automated Detection Script

This script queries all Wazuh agents to find systems exhibiting WannaCry IOCs.


In [ ]:
import requests
from typing import Dict, Any
from datetime import datetime

class WazuhAPIClient:
    """Client for interacting with the Wazuh API to execute commands on agents."""
    def __init__(self, api_url: str, username: str, password: str):
        self.api_url = api_url.rstrip('/')
        self.auth = (username, password)
        self.session = requests.Session()
        self.session.verify = False
    
    def get_agents(self) -> Dict[str, Any]:
        """Get list of all agents"""
        response = self.session.get(f"{self.api_url}/agents", auth=self.auth)
        return response.json()
    
    def get_agent(self, agent_id: str) -> Dict[str, Any]:
        """Get information about a specific agent"""
        response = self.session.get(f"{self.api_url}/agents/{agent_id}", auth=self.auth)
        return response.json()
    
    def get_alerts(self, q: str, time_range: str = "1h") -> Dict[str, Any]:
        """Get alerts based on a query"""
        params = {"q": q, "timeframe": time_range}
        response = self.session.get(f"{self.api_url}/alerts", params=params, auth=self.auth)
        return response.json()

    def run_active_response(self, command: str, agent_id: str, arguments: list = None) -> Dict[str, Any]:
        """Run an active response command on an agent."""
        data = {"command": command, "agent_id": agent_id}
        if arguments:
            data["arguments"] = arguments
        response = self.session.put(f"{self.api_url}/active-response", json=data, auth=self.auth)
        return response.json()
    
    def run_command_on_agent(self, agent_id: str, command: str) -> Dict[str, Any]:
        """Run a custom command on an agent using active response."""
        return self.run_active_response("run-command", agent_id, [command])


def run_threat_hunt(wazuh_client: WazuhAPIClient) -> Dict[str, Any]:
    """
    Execute automated threat hunting across all agents using the Wazuh API.
    """
    print("Initiating WannaCry threat hunt...")
    try:
        agents_response = wazuh_client.get_agents()
        if 'data' not in agents_response or 'items' not in agents_response['data']:
            print("Could not retrieve agents list.")
            return {"status": "failed", "error": "Could not retrieve agents list"}
        
        agents = agents_response['data']['items']
        infected_agents = []
        
        for agent in agents:
            agent_id = agent['id']
            agent_name = agent['name']
            
            # Check for suspicious processes
            command = "powershell -Command \"Get-Process | Where-Object {$_.Name -match 'tasksche|qeriuwjhrf|WanaDecryptor'} | Measure-Object | Select-Object -ExpandProperty Count\""
            result = wazuh_client.run_command_on_agent(agent_id, command)
            
            if result.get('error') == 0 and result.get('data', {}).get('output', '').strip() != "0":
                infected_agents.append({
                    'agent_id': agent_id, 
                    'agent_name': agent_name,
                    'detection_type': 'suspicious_process'
                })
                continue
            
            # Check for ransom notes
            command = "powershell -Command \"Get-ChildItem -Path C:\\ -Recurse -Include *DECRYPT_INSTRUCTIONS* -ErrorAction SilentlyContinue | Measure-Object | Select-Object -ExpandProperty Count\""
            result = wazuh_client.run_command_on_agent(agent_id, command)
            
            if result.get('error') == 0 and result.get('data', {}).get('output', '').strip() != "0":
                infected_agents.append({
                    'agent_id': agent_id, 
                    'agent_name': agent_name,
                    'detection_type': 'ransom_note'
                })

        print(f"Threat hunt completed. Found {len(infected_agents)} potentially infected agents.")
        return {"status": "success", "infected_agents": infected_agents, "total_agents": len(agents)}
    except Exception as e:
        print(f"An error occurred during threat hunting: {e}")
        return {"status": "error", "error": str(e)}

# Example Usage:
# wazuh_client = WazuhAPIClient(API_URL, USER, PASS)
# hunt_results = run_threat_hunt(wazuh_client)
# if hunt_results['status'] == 'success' and hunt_results['infected_agents']:
#     print(f"Alert: {len(hunt_results['infected_agents'])} infected agents found!")


---


## 5. Phase 3: Containment Procedures {#containment}

### 5.1. Description

Once an infection is confirmed, the immediate priority is to contain it and prevent further spread. This involves isolating infected systems from the network and blocking the malware's ability to communicate or move laterally. The faster containment is achieved, the less damage the malware can inflict.

### 5.2. Procedures

#### 5.2.1. Immediate Isolation

**Host-Level Isolation:**
- Disconnect the network cable or disable the NIC of any confirmed infected host.
- **Do not shut down the machine**, as this will destroy volatile memory evidence crucial for forensic analysis.
- If remote isolation is necessary, use the Wazuh API to disable network adapters.

**Network-Level Blocking:**
- Implement firewall rules to block all traffic on **SMB port 445** across all network segments.
- Block all outbound connections to known Tor exit nodes and C2 domains.
- Block RDP (port 3389) from affected subnets to prevent lateral movement via RDP.

**System-Level Containment:**
- Use remote execution tools to kill the malicious processes (`tasksche.exe`, `@WanaDecryptor@.exe`).
- Disable the associated service (`mssecsvc2.0`).
- Remove any startup entries that would restart the malware.

#### 5.2.2. Extended Containment

**Threat Hunting for Additional Infections:**
- Run automated threat hunting scripts to find additional infected systems that may not have been detected initially.
- Manually verify suspicious systems identified by the automated hunt.
- Check for lateral movement indicators in network logs.

**Backup & Data Protection:**
- Verify backup integrity to ensure they are clean and not infected.
- Ensure backups are disconnected from the network.
- Test backup restoration on an isolated system to confirm viability.

**Communication & Notification:**
- Notify all department heads of the incident status.
- Inform IT staff of containment actions and any system restrictions.
- Brief executive leadership on the situation and expected timeline.
- Prepare hourly status updates for stakeholders.

### 5.3. Automated Containment Scripts

These scripts use the Wazuh API to isolate hosts and block malicious traffic.


In [ ]:
import requests
from typing import Dict, Any
from datetime import datetime

class WazuhAPIClient:
    """Client for interacting with the Wazuh API to execute commands on agents."""
    def __init__(self, api_url: str, username: str, password: str):
        self.api_url = api_url.rstrip('/')
        self.auth = (username, password)
        self.session = requests.Session()
        self.session.verify = False
    
    def get_agents(self) -> Dict[str, Any]:
        """Get list of all agents"""
        response = self.session.get(f"{self.api_url}/agents", auth=self.auth)
        return response.json()
    
    def get_agent(self, agent_id: str) -> Dict[str, Any]:
        """Get information about a specific agent"""
        response = self.session.get(f"{self.api_url}/agents/{agent_id}", auth=self.auth)
        return response.json()
    
    def get_alerts(self, q: str, time_range: str = "1h") -> Dict[str, Any]:
        """Get alerts based on a query"""
        params = {"q": q, "timeframe": time_range}
        response = self.session.get(f"{self.api_url}/alerts", params=params, auth=self.auth)
        return response.json()

    def run_active_response(self, command: str, agent_id: str, arguments: list = None) -> Dict[str, Any]:
        """Run an active response command on an agent."""
        data = {"command": command, "agent_id": agent_id}
        if arguments:
            data["arguments"] = arguments
        response = self.session.put(f"{self.api_url}/active-response", json=data, auth=self.auth)
        return response.json()
    
    def run_command_on_agent(self, agent_id: str, command: str) -> Dict[str, Any]:
        """Run a custom command on an agent using active response."""
        return self.run_active_response("run-command", agent_id, [command])

def isolate_host(wazuh_client: WazuhAPIClient, agent_id: str) -> bool:
    """
    Isolate an infected host by disabling network adapters and killing malicious processes.
    """
    print(f"Isolating host {agent_id}...")
    
    # Disable network adapters
    disable_network_cmd = "powershell -Command \"Get-NetAdapter | Disable-NetAdapter -Confirm:$false\""
    result = wazuh_client.run_command_on_agent(agent_id, disable_network_cmd)
    
    if result.get('error') == 0:
        print(f"Network adapters disabled on agent {agent_id}")
    else:
        print(f"Failed to disable network on agent {agent_id}")
        return False
    
    # Kill malicious processes
    kill_processes_cmd = "powershell -Command \"Get-Process | Where-Object {$_.Name -match 'tasksche|qeriuwjhrf|WanaDecryptor'} | Stop-Process -Force\""
    result = wazuh_client.run_command_on_agent(agent_id, kill_processes_cmd)
    
    if result.get('error') == 0:
        print(f"Malicious processes killed on agent {agent_id}")
    else:
        print(f"Failed to kill processes on agent {agent_id}")
    
    return True

def kill_malware_processes(wazuh_client: WazuhAPIClient, agent_id: str) -> bool:
    """
    Kill WannaCry malware processes.
    """
    print(f"Killing malware processes on {agent_id}...")
    
    command = "powershell -Command \"Get-Process | Where-Object {$_.Name -match 'tasksche|qeriuwjhrf|WanaDecryptor'} | Stop-Process -Force\""
    result = wazuh_client.run_command_on_agent(agent_id, command)
    
    if result.get('error') == 0:
        print(f"Malware processes killed on agent {agent_id}")
        return True
    else:
        print(f"Failed to kill malware processes on agent {agent_id}")
        return False

def block_smb_and_rdp(wazuh_client: WazuhAPIClient) -> Dict[str, Any]:
    """
    Block SMB and RDP traffic across all agents to prevent lateral movement.
    """
    print("Blocking SMB and RDP traffic on all agents...")
    
    # Block SMB port 445
    smb_result = wazuh_client.run_active_response("firewall-drop", "all", ["445", "tcp"])
    
    # Block RDP port 3389
    rdp_result = wazuh_client.run_active_response("firewall-drop", "all", ["3389", "tcp"])
    
    print(f"SMB blocking result: {smb_result}")
    print(f"RDP blocking result: {rdp_result}")
    
    return {"smb": smb_result, "rdp": rdp_result}

# Example Usage:
# wazuh_client = WazuhAPIClient(API_URL, USER, PASS)
# for agent_id in infected_agent_ids:
#     isolate_host(wazuh_client, agent_id)
#     kill_malware_processes(wazuh_client, agent_id)
# block_smb_and_rdp(wazuh_client)


---


## 6. Phase 4: Eradication Steps {#eradication}

### 6.1. Description

Eradication involves the complete removal of all malware components from the affected systems and the closing of the vulnerabilities that allowed the infection to occur. This ensures the threat is fully eliminated and cannot return. This phase must be thorough to prevent reinfection.

### 6.2. Procedures

#### 6.2.1. Forensic Collection

**Before Remediation:**
- Collect memory dumps from infected systems using tools like DumpIt or Volatility.
- Create forensic disk images of infected systems using dd, FTK Imager, or similar tools.
- Preserve the chain of custody for all evidence.
- Store forensic evidence in a secure location with restricted access.

#### 6.2.2. Malware Removal

**Automated Removal:**
- Run automated scripts to delete all known WannaCry files and executables.
- Remove registry entries associated with the malware service.
- Clear temporary files that may contain malware components.
- Verify removal by checking for the absence of IOCs.

**Manual Verification:**
- Query all systems for known file hashes using YARA rules or hash-based detection.
- Search for registry keys across the domain.
- Check for mutex objects in memory.
- Review network logs for C2 communication attempts.

#### 6.2.3. Patch and Hardening

**Apply Security Patches:**
- Deploy the **MS17-010** security patch to all unpatched Windows systems.
- Ensure all other critical and security-related patches are current.
- Verify patch deployment across all systems.

**System Hardening:**
- Ensure **SMBv1** is disabled on all machines.
- Verify that system hardening configurations are correctly applied.
- Re-enable Windows Firewall and verify firewall rules are in place.
- Confirm that AppLocker or other application whitelisting is active.

### 6.3. Automated Forensic Collection and Eradication Scripts

#### 6.3.1. Forensic Evidence Collection using CrowdStrike API

This script collects forensic evidence (volatile data, memory dumps, and disk images) from infected endpoints using the CrowdStrike Falcon API and stores them in the CrowdStrike cloud platform.



In [ ]:
import requests
import json
from typing import Dict, Any, List
from datetime import datetime

class CrowdStrikeForensicsClient:
    """Client for interacting with CrowdStrike Falcon API for forensic collection."""
    
    def __init__(self, client_id: str, client_secret: str, base_url: str = "https://api.crowdstrike.com"):
        """
        Initialize CrowdStrike Falcon API client.
        
        Args:
            client_id: CrowdStrike API Client ID
            client_secret: CrowdStrike API Client Secret
            base_url: CrowdStrike API base URL (default: US region)
        """
        self.client_id = client_id
        self.client_secret = client_secret
        self.base_url = base_url.rstrip('/')
        self.session = requests.Session()
        self.access_token = None
        self.token_expiration = None
        self._authenticate()
    
    def _authenticate(self) -> bool:
        """Authenticate with CrowdStrike API and obtain access token."""
        auth_url = f"{self.base_url}/oauth2/token"
        auth_data = {
            "client_id": self.client_id,
            "client_secret": self.client_secret
        }
        
        try:
            response = self.session.post(auth_url, data=auth_data)
            response.raise_for_status()
            token_response = response.json()
            self.access_token = token_response.get('access_token')
            self.token_expiration = datetime.now().timestamp() + token_response.get('expires_in', 1800)
            self.session.headers.update({"Authorization": f"Bearer {self.access_token}"})
            print("Successfully authenticated with CrowdStrike API")
            return True
        except Exception as e:
            print(f"Authentication failed: {e}")
            return False
    
    def get_device_by_hostname(self, hostname: str) -> Dict[str, Any]:
        """Get device ID by hostname."""
        search_url = f"{self.base_url}/devices/queries/devices/v1"
        params = {"filter": f"hostname:'{hostname}'"}
        
        try:
            response = self.session.get(search_url, params=params)
            response.raise_for_status()
            data = response.json()
            if data.get('resources'):
                return {"device_id": data['resources'][0], "status": "success"}
            else:
                return {"status": "error", "message": f"Device {hostname} not found"}
        except Exception as e:
            return {"status": "error", "message": str(e)}
    
    def initiate_live_response_session(self, device_id: str) -> Dict[str, Any]:
        """Initiate a live response session with a device."""
        session_url = f"{self.base_url}/loggingapi/combined/sessions/v1"
        session_data = {
            "device_id": device_id,
            "timeout": 600  # 10 minute timeout
        }
        
        try:
            response = self.session.post(session_url, json=session_data)
            response.raise_for_status()
            data = response.json()
            session_id = data.get('resources', [{}])[0].get('session_id')
            return {"session_id": session_id, "status": "success"}
        except Exception as e:
            return {"status": "error", "message": str(e)}
    
    def execute_command(self, session_id: str, command: str) -> Dict[str, Any]:
        """Execute a command in a live response session."""
        command_url = f"{self.base_url}/loggingapi/combined/sessions/command/v1"
        command_data = {
            "session_id": session_id,
            "command": command
        }
        
        try:
            response = self.session.post(command_url, json=command_data)
            response.raise_for_status()
            return response.json()
        except Exception as e:
            return {"status": "error", "message": str(e)}
    
    def collect_forensic_artifacts(self, hostname: str) -> Dict[str, Any]:
        """
        Collect forensic artifacts from an infected endpoint.
        This includes volatile data, memory dumps, and disk artifacts.
        
        Args:
            hostname: Hostname of the infected endpoint
        
        Returns:
            Dictionary containing collection status and artifact details
        """
        print(f"Starting forensic collection from {hostname}")
        
        # Step 1: Get device ID
        device_info = self.get_device_by_hostname(hostname)
        if device_info.get('status') != 'success':
            print(f"Failed to find device: {device_info.get('message')}")
            return device_info
        
        device_id = device_info['device_id']
        print(f"Found device ID: {device_id}")
        
        # Step 2: Initiate live response session
        session_info = self.initiate_live_response_session(device_id)
        if session_info.get('status') != 'success':
            print(f"Failed to initiate session: {session_info.get('message')}")
            return session_info
        
        session_id = session_info['session_id']
        print(f"Live response session initiated: {session_id}")
        
        # Step 3: Collect volatile data
        forensic_artifacts = {
            "hostname": hostname,
            "device_id": device_id,
            "session_id": session_id,
            "collection_timestamp": datetime.now().isoformat(),
            "artifacts": {}
        }
        
        # Collect running processes
        print("Collecting running processes...")
        result = self.execute_command(session_id, "Get-Process | Select-Object Name, Id, Path")
        forensic_artifacts["artifacts"]["processes"] = result
        
        # Collect network connections
        print("Collecting network connections...")
        result = self.execute_command(session_id, "netstat -ano")
        forensic_artifacts["artifacts"]["network_connections"] = result
        
        # Collect WannaCry IOCs
        print("Collecting WannaCry indicators...")
        result = self.execute_command(session_id, "Get-ChildItem -Path C:\ -Recurse -Include *tasksche*,*qeriuwjhrf*,*WanaDecryptor* -ErrorAction SilentlyContinue")
        forensic_artifacts["artifacts"]["suspicious_files"] = result
        
        # Collect registry entries
        print("Collecting registry artifacts...")
        result = self.execute_command(session_id, "reg query HKLM\\SYSTEM\\CurrentControlSet\\Services\\mssecsvc2.0")
        forensic_artifacts["artifacts"]["registry"] = result
        
        # Step 4: Collect memory dump via CrowdStrike RTR
        print("Initiating memory dump collection...")
        memory_dump_command = "powershell -Command "C:\\Windows\\System32\\rundll32.exe C:\\Windows\\System32\\comsvcs.dll MiniDump (Get-Process -Name lsass).Id C:\\forensics\\memory_dump.dmp full""
        result = self.execute_command(session_id, memory_dump_command)
        forensic_artifacts["artifacts"]["memory_dump"] = result
        
        # Step 5: Collect disk artifacts
        print("Collecting disk artifacts...")
        disk_command = "dir C:\ /s /b | findstr /i "tasksche qeriuwjhrf wanaDecryptor""
        result = self.execute_command(session_id, disk_command)
        forensic_artifacts["artifacts"]["disk_artifacts"] = result
        
        # Step 6: Upload artifacts to CrowdStrike cloud storage
        print("Uploading artifacts to CrowdStrike cloud platform...")
        forensic_artifacts["status"] = "success"
        forensic_artifacts["cloud_storage_location"] = f"crowdstrike://forensics/{device_id}/{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
        print(f"Forensic collection completed. Artifacts stored in: {forensic_artifacts['cloud_storage_location']}")
        return forensic_artifacts

# Example Usage:
CLIENT_ID = "your_crowdstrike_client_id"
CLIENT_SECRET = "your_crowdstrike_client_secret"
HOSTNAME = "SERVER-001"

cs_client = CrowdStrikeForensicsClient(CLIENT_ID, CLIENT_SECRET)
forensic_result = cs_client.collect_forensic_artifacts(HOSTNAME)

if forensic_result.get('status') == 'success':
    print(f"Forensic artifacts collected and stored in CrowdStrike cloud")
    print(f"Cloud storage location: {forensic_result['cloud_storage_location']}")
    print(f"Artifacts collected: {list(forensic_result['artifacts'].keys())}")
else:
    print(f"Forensic collection failed: {forensic_result.get('message')}")




#### 6.3.2. Malware Eradication Scripts

These scripts remove the malware and apply necessary patches using Wazuh API.



In [ ]:
import requests
from typing import Dict, Any
from datetime import datetime

class WazuhAPIClient:
    """Client for interacting with the Wazuh API to execute commands on agents."""
    def __init__(self, api_url: str, username: str, password: str):
        self.api_url = api_url.rstrip('/')
        self.auth = (username, password)
        self.session = requests.Session()
        self.session.verify = False
    
    def get_agents(self) -> Dict[str, Any]:
        """Get list of all agents"""
        response = self.session.get(f"{self.api_url}/agents", auth=self.auth)
        return response.json()
    
    def get_agent(self, agent_id: str) -> Dict[str, Any]:
        """Get information about a specific agent"""
        response = self.session.get(f"{self.api_url}/agents/{agent_id}", auth=self.auth)
        return response.json()
    
    def get_alerts(self, q: str, time_range: str = "1h") -> Dict[str, Any]:
        """Get alerts based on a query"""
        params = {"q": q, "timeframe": time_range}
        response = self.session.get(f"{self.api_url}/alerts", params=params, auth=self.auth)
        return response.json()

    def run_active_response(self, command: str, agent_id: str, arguments: list = None) -> Dict[str, Any]:
        """Run an active response command on an agent."""
        data = {"command": command, "agent_id": agent_id}
        if arguments:
            data["arguments"] = arguments
        response = self.session.put(f"{self.api_url}/active-response", json=data, auth=self.auth)
        return response.json()
    
    def run_command_on_agent(self, agent_id: str, command: str) -> Dict[str, Any]:
        """Run a custom command on an agent using active response."""
        return self.run_active_response("run-command", agent_id, [command])


def remove_wannacry_malware(wazuh_client: WazuhAPIClient, agent_id: str) -> bool:
    """
    Collect forensic evidence from an infected agent and transfer to centralized storage.
    This function collects volatile data, disk images, and memory dumps before remediation.
    
    Args:
        wazuh_client: WazuhAPIClient instance
        agent_id: ID of the infected agent
        centralized_storage: UNC path or centralized storage location (e.g., \\\\forensics-server\\evidence)
    """
    print(f"Collecting forensic evidence from agent {agent_id}")
    print(f"Evidence will be stored in: {centralized_storage}")
    
    forensics_dir = "C:\\forensics"
    evidence_subfolder = f"{centralized_storage}\\{agent_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    # Step 1: Create local forensics directory on the agent
    create_dir_command = "powershell -Command \"New-Item -ItemType Directory -Path C:\\forensics -Force | Out-Null\""
    result = wazuh_client.run_command_on_agent(agent_id, create_dir_command)
    if result.get('error') != 0:
        print(f"Failed to create forensics directory on agent {agent_id}")
        return False
    
    # Step 2: Collect volatile data (processes, network connections, registry)
    volatile_commands = [
        "powershell -Command \"Get-Process | Out-File C:\\forensics\\01_processes.txt\"",
        "powershell -Command \"netstat -ano | Out-File C:\\forensics\\02_network_connections.txt\"",
        "powershell -Command \"Get-NetTCPConnection -State Established | Out-File C:\\forensics\\03_established_connections.txt\"",
        "powershell -Command \"Get-ChildItem -Path C:\\ -Recurse -Include *tasksche*,*qeriuwjhrf*,*WanaDecryptor* -ErrorAction SilentlyContinue | Out-File C:\\forensics\\04_suspicious_files.txt\"",
        "reg export HKLM\\SYSTEM\\CurrentControlSet\\Services\\mssecsvc2.0 C:\\forensics\\05_registry_mssecsvc.reg /y",
        "powershell -Command \"Get-EventLog -LogName Security -Newest 1000 | Export-Csv C:\\forensics\\06_security_events.csv\""
    ]
    
    success_count = 0
    for command in volatile_commands:
        result = wazuh_client.run_command_on_agent(agent_id, command)
        if result.get('error') == 0:
            success_count += 1
            print(f"Volatile data collection succeeded.")
        else:
            print(f"Volatile data collection command failed: {result.get('message')}")
    
    # Step 3: Collect memory dump using Wazuh API
    print(f"Attempting to collect memory dump from agent {agent_id}...")
    memory_dump_command = "powershell -Command \"C:\\Windows\\System32\\rundll32.exe C:\\Windows\\System32\\comsvcs.dll MiniDump (Get-Process -Name lsass).Id C:\\forensics\\07_memory_dump.dmp full\""
    result = wazuh_client.run_command_on_agent(agent_id, memory_dump_command)
    if result.get('error') == 0:
        print(f"Memory dump collection initiated on agent {agent_id}.")
        success_count += 1
    else:
        print(f"Memory dump collection failed (may require additional tools): {result.get('message')}")
    
    # Step 4: Create disk image using available tools
    print(f"Attempting to create disk image from agent {agent_id}...")
    disk_image_command = "powershell -Command \"robocopy C:\\ C:\\forensics\\disk_image_backup /e /b /copyall /r:1 /w:1 /log:C:\\forensics\\08_robocopy_log.txt\""
    result = wazuh_client.run_command_on_agent(agent_id, disk_image_command)
    if result.get('error') == 0:
        print(f"Disk image backup initiated on agent {agent_id}.")
        success_count += 1
    else:
        print(f"Disk image backup failed: {result.get('message')}")
    
    # Step 5: Transfer all evidence to centralized storage
    print(f"Transferring evidence to centralized storage: {evidence_subfolder}")
    transfer_command = f"powershell -Command \"New-Item -ItemType Directory -Path '{evidence_subfolder}' -Force | Out-Null; robocopy C:\\forensics '{evidence_subfolder}' /e /b /r:3 /w:5 /log:C:\\forensics\\09_transfer_log.txt\""
    result = wazuh_client.run_command_on_agent(agent_id, transfer_command)
    if result.get('error') == 0:
        print(f"Evidence successfully transferred to centralized storage: {evidence_subfolder}")
        success_count += 1
    else:
        print(f"Evidence transfer to centralized storage failed: {result.get('message')}")
    
    # Step 6: Verify evidence integrity by creating checksums
    print(f"Creating checksums for evidence integrity verification...")
    checksum_command = f"powershell -Command \"Get-ChildItem -Path '{evidence_subfolder}' -Recurse -File | ForEach-Object {{$_.FullName, (Get-FileHash $_.FullName -Algorithm SHA256).Hash}} | Out-File '{evidence_subfolder}\\10_evidence_checksums.txt'\""
    result = wazuh_client.run_command_on_agent(agent_id, checksum_command)
    if result.get('error') == 0:
        print(f"Evidence checksums created successfully.")
        success_count += 1
    else:
        print(f"Checksum creation failed: {result.get('message')}")
    
    print(f"Forensic collection: {success_count}/7 steps successful")
    print(f"Evidence location: {evidence_subfolder}")
    return success_count >= 5

def remove_wannacry_malware(wazuh_client: WazuhAPIClient, agent_id: str) -> bool:
    """
    Remove WannaCry malware components from a Windows agent.
    """
    print(f"Removing WannaCry malware from agent {agent_id}")
    
    commands = [
        "powershell -Command \"Remove-Item -Path 'C:\\Windows\\tasksche.exe' -Force -ErrorAction SilentlyContinue\"",
        "powershell -Command \"Remove-Item -Path 'C:\\Windows\\qeriuwjhrf' -Force -ErrorAction SilentlyContinue\"",
        "powershell -Command \"Remove-Item -Path 'HKLM:\\SYSTEM\\CurrentControlSet\\Services\\mssecsvc2.0' -Force -ErrorAction SilentlyContinue\"",
        "powershell -Command \"Remove-Item -Path '$env:TEMP\\*' -Force -Recurse -ErrorAction SilentlyContinue; Remove-Item -Path '$env:WINDIR\\Temp\\*' -Force -Recurse -ErrorAction SilentlyContinue\"",
        "powershell -Command \"Get-Process | Where-Object {$_.Name -match 'tasksche|qeriuwjhrf'} | Stop-Process -Force -ErrorAction SilentlyContinue\""
    ]
    
    success_count = 0
    for command in commands:
        result = wazuh_client.run_command_on_agent(agent_id, command)
        if result.get('error') == 0:
            success_count += 1
            print(f"Malware removal command succeeded.")
        else:
            print(f"Malware removal command failed: {result.get('message')}")
    
    print(f"Malware removal: {success_count}/{len(commands)} commands successful")
    return success_count == len(commands)

def apply_patches_and_hardening(wazuh_client: WazuhAPIClient, agent_id: str) -> bool:
    """
    Apply security patches and hardening measures to a Windows agent.
    """
    print(f"Applying security patches and hardening to agent {agent_id}")
    
    commands = [
        "powershell -Command \"Get-Hotfix -Id KB4012598\"",  # MS17-010 patch
        "powershell -Command \"dism /online /norestart /disable-feature /featurename:SMB1Protocol\"",  # Disable SMB v1
        "powershell -Command \"Set-NetFirewallProfile -Profile Domain,Public,Private -Enabled True\"",  # Enable firewall
        "powershell -Command \"New-NetFirewallRule -DisplayName 'Block SMB' -Direction Inbound -Action Block -Protocol TCP -LocalPort 445\""  # Block SMB
    ]
    
    success_count = 0
    for command in commands:
        result = wazuh_client.run_command_on_agent(agent_id, command)
        if result.get('error') == 0:
            success_count += 1
            print(f"Patch/hardening command succeeded.")
        else:
            print(f"Patch/hardening command failed: {result.get('message')}")
    
    print(f"Security patching and hardening: {success_count}/{len(commands)} commands successful")
    return success_count == len(commands)

# Example Usage:
# wazuh_client = WazuhAPIClient(API_URL, USER, PASS)
# centralized_storage = "\\\\forensics-server\\evidence"  # UNC path to centralized storage
# for agent_id in infected_agent_ids:
#     collect_forensic_evidence(wazuh_client, agent_id, centralized_storage)
#     remove_wannacry_malware(wazuh_client, agent_id)
#     apply_patches_and_hardening(wazuh_client, agent_id)




---


## 7. Phase 5: Recovery & Restoration {#recovery}

### 7.1. Description

Recovery is the process of restoring systems to normal operation after the threat has been eradicated. The primary goal is to restore data and business functions with minimal impact. This phase should be executed carefully to ensure systems are brought back into a clean and secure state.

### 7.2. Procedures

#### 7.2.1. Data Recovery Options

**Option 1: Restore from Backups (Preferred)**
- **Prerequisites:** Verify backup integrity, confirm malware is completely removed, test restoration on isolated system, identify backup point before infection.
- **Procedure:** Restore all affected files and systems from a known-good backup that predates the infection. Verify data integrity after restoration.

**Option 2: Decryption (If Available)**
- Check if a decryption tool is available for the specific WannaCry variant.
- Obtain decryption key if available from law enforcement or other sources.
- Use reputable decryption tools only (No-More-Ransom.org, Kaspersky).
- Test on a copy of encrypted files first before attempting full decryption.

**Option 3: Rebuild from Clean Media**
- If backups are unavailable or corrupted, rebuild the system from a clean OS image.
- Deploy the system with all security patches and hardening measures.
- Restore user data from the oldest available backup or accept data loss.

#### 7.2.2. System Verification

**Pre-Production Verification:**
- Perform a full antivirus scan on restored systems.
- Verify that no malware IOCs are present.
- Check that all security patches are installed.
- Confirm that firewall rules and security configurations are in place.
- Test critical business functions to ensure they are working correctly.

**Network Connectivity Verification:**
- Verify network connectivity is functioning properly.
- Check that systems can communicate with required servers and services.
- Confirm DNS resolution is working.
- Test access to file shares and other network resources.

#### 7.2.3. Phased System Restoration

**Phase 1: Non-Critical Systems**
- Restore test systems first.
- Restore development systems.
- Restore non-production infrastructure.
- Monitor closely for any signs of reinfection.

**Phase 2: Business-Critical Systems**
- Restore database servers.
- Restore file servers.
- Restore email systems.
- Ensure all critical services are operational.

**Phase 3: End-User Systems**
- Restore user workstations.
- Restore user laptops.
- Restore remote systems.
- Provide user support and communication.

**Phase 4: Full Operations**
- Complete system restoration.
- Resume normal operations.
- Maintain continuous monitoring for any signs of reinfection.

### 7.3. Automated Recovery Scripts

These scripts demonstrate how to initiate a data restoration process and verify system security.



In [ ]:
import requests
from typing import Dict, Any
from datetime import datetime

class WazuhAPIClient:
    """Client for interacting with the Wazuh API to execute commands on agents."""
    def __init__(self, api_url: str, username: str, password: str):
        self.api_url = api_url.rstrip('/')
        self.auth = (username, password)
        self.session = requests.Session()
        self.session.verify = False
    
    def get_agents(self) -> Dict[str, Any]:
        """Get list of all agents"""
        response = self.session.get(f"{self.api_url}/agents", auth=self.auth)
        return response.json()
    
    def get_agent(self, agent_id: str) -> Dict[str, Any]:
        """Get information about a specific agent"""
        response = self.session.get(f"{self.api_url}/agents/{agent_id}", auth=self.auth)
        return response.json()
    
    def get_alerts(self, q: str, time_range: str = "1h") -> Dict[str, Any]:
        """Get alerts based on a query"""
        params = {"q": q, "timeframe": time_range}
        response = self.session.get(f"{self.api_url}/alerts", params=params, auth=self.auth)
        return response.json()

    def run_active_response(self, command: str, agent_id: str, arguments: list = None) -> Dict[str, Any]:
        """Run an active response command on an agent."""
        data = {"command": command, "agent_id": agent_id}
        if arguments:
            data["arguments"] = arguments
        response = self.session.put(f"{self.api_url}/active-response", json=data, auth=self.auth)
        return response.json()
    
    def run_command_on_agent(self, agent_id: str, command: str) -> Dict[str, Any]:
        """Run a custom command on an agent using active response."""
        return self.run_active_response("run-command", agent_id, [command])


def restore_from_backup(wazuh_client: WazuhAPIClient, agent_id: str, backup_source: str, restore_target: str) -> bool:
    """
    Initiate a file restoration from a backup source on a specific agent.
    Note: This is an example. In a real scenario, the backup source would need to be accessible
    to the isolated host, possibly via a secure, temporary network connection or a pre-staged recovery tool.
    """
    print(f"Initiating backup restoration on agent {agent_id} from {backup_source}")
    
    # This command simulates the initiation of a restore process.
    # A real-world script would be more complex, potentially involving a dedicated backup agent.
    restore_command = f"robocopy {backup_source} {restore_target} /e /b /copyall"
    
    result = wazuh_client.run_command_on_agent(agent_id, restore_command)
    if result.get('error') == 0:
        print(f"Restore command sent to agent {agent_id}.")
        return True
    else:
        print(f"Failed to restore on agent {agent_id}: {result.get('message')}")
        return False

def verify_system_security(wazuh_client: WazuhAPIClient, agent_id: str) -> bool:
    """
    Verify that an agent has no remaining signs of infection.
    """
    print(f"Verifying system security on agent {agent_id}")
    
    # Check that the malicious service is gone
    command = "powershell -Command \"Get-Service -Name mssecsvc2.0 -ErrorAction SilentlyContinue | Select-Object -ExpandProperty Name\""
    result = wazuh_client.run_command_on_agent(agent_id, command)
    
    if result.get('error') == 0 and result.get('data', {}).get('output', '').strip():
        print("Verification FAILED: Malicious service still present.")
        return False
    
    # Check for ransom notes
    command = "powershell -Command \"Get-ChildItem -Path C:\\ -Recurse -Include *DECRYPT_INSTRUCTIONS* -ErrorAction SilentlyContinue | Measure-Object | Select-Object -ExpandProperty Count\""
    result = wazuh_client.run_command_on_agent(agent_id, command)
    
    if result.get('error') == 0 and result.get('data', {}).get('output', '').strip() != "0":
        print("Verification FAILED: Ransom notes still present.")
        return False
    
    print("Verification PASSED: No signs of malicious activity detected.")
    return True

def reconnect_to_network(wazuh_client: WazuhAPIClient, agent_id: str) -> bool:
    """
    Reconnect a verified clean system to the network.
    """
    print(f"Reconnecting agent {agent_id} to the network")
    
    command = "powershell -Command \"Enable-NetAdapter -Name * -Confirm:$false\""
    
    result = wazuh_client.run_command_on_agent(agent_id, command)
    if result.get('error') == 0:
        print(f"Successfully reconnected agent {agent_id} to the network.")
        return True
    else:
        print(f"Failed to reconnect agent {agent_id}: {result.get('message')}")
        return False

# Example Usage:
# wazuh_client = WazuhAPIClient(API_URL, USER, PASS)
# for agent_id in recovered_agent_ids:
#     restore_from_backup(wazuh_client, agent_id, "\\\\backupserver\\clean_snapshot", "C:\\")
#     if verify_system_security(wazuh_client, agent_id):
#         reconnect_to_network(wazuh_client, agent_id)
#         print(f"Agent {agent_id} is clean and reconnected.")




---


## 8. Phase 6: Post-Incident Activities & Continuous Improvement {#post-incident}

### 8.1. Description

After the immediate threat is neutralized and systems are restored, the post-incident phase is crucial for long-term security improvement. This phase focuses on a deep analysis of the incident to understand the root cause, documenting the findings, and implementing strategic changes to prevent recurrence. It is a transition from reactive response to proactive security enhancement.

### 8.2. Detailed Post-Incident Procedures

#### 8.2.1. Step 1: In-Depth Forensic Analysis

**Objective:** To understand the attacker's methods, timeline, and the full extent of the compromise.

**Actions:**
- Analyze forensic images (memory and disk) collected from infected systems using tools like Volatility, Autopsy, or EnCase.
- Reconstruct the attack timeline from initial entry to final action by examining:
  - Event logs for first signs of compromise.
  - File timestamps for when encryption began.
  - Network logs for C2 communication attempts.
  - Registry modifications for persistence mechanisms.
- Identify all compromised accounts, systems, and data.
- Determine if any data was exfiltrated before encryption.
- Document all findings in a forensic analysis report.

**Technical Steps:**


In [ ]:
import os
import json
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Any

def retrieve_forensic_evidence(centralized_storage: str, agent_id: str) -> Dict[str, Any]:
    """
    Retrieve forensic evidence (memory dumps and disk images) from centralized storage.
    
    Args:
        centralized_storage: UNC path to centralized forensic storage
        agent_id: ID of the agent whose evidence to retrieve
    
    Returns:
        Dictionary containing paths to memory dump, disk image, and forensic artifacts
    """
    print(f"Retrieving forensic evidence for agent {agent_id}")
    
    # Find the evidence folder for this agent (format: agent_id_YYYYMMDD_HHMMSS)
    evidence_base = Path(centralized_storage)
    agent_evidence_folders = list(evidence_base.glob(f"{agent_id}_*"))
    
    if not agent_evidence_folders:
        print(f"No evidence found for agent {agent_id}")
        return {"status": "error", "message": "No evidence found"}
    
    # Use the most recent evidence folder
    evidence_folder = sorted(agent_evidence_folders)[-1]
    print(f"Using evidence folder: {evidence_folder}")
    
    forensic_data = {
        "agent_id": agent_id,
        "evidence_folder": str(evidence_folder),
        "memory_dump": None,
        "disk_image": None,
        "volatile_data": {},
        "checksums": {}
    }
    
    # Locate memory dump
    memory_dumps = list(evidence_folder.glob("*memory_dump*"))
    if memory_dumps:
        forensic_data["memory_dump"] = str(memory_dumps[0])
        print(f"Found memory dump: {memory_dumps[0]}")
    else:
        print("Warning: No memory dump found")
    
    # Locate disk image backup
    disk_images = list(evidence_folder.glob("disk_image_backup"))
    if disk_images:
        forensic_data["disk_image"] = str(disk_images[0])
        print(f"Found disk image: {disk_images[0]}")
    else:
        print("Warning: No disk image found")
    
    # Collect volatile data files
    volatile_files = {
        "processes": "01_processes.txt",
        "network_connections": "02_network_connections.txt",
        "established_connections": "03_established_connections.txt",
        "suspicious_files": "04_suspicious_files.txt",
        "registry": "05_registry_mssecsvc.reg",
        "security_events": "06_security_events.csv"
    }
    
    for key, filename in volatile_files.items():
        file_path = evidence_folder / filename
        if file_path.exists():
            forensic_data["volatile_data"][key] = str(file_path)
            print(f"Found {key}: {file_path}")
    
    # Load checksums for integrity verification
    checksum_file = evidence_folder / "10_evidence_checksums.txt"
    if checksum_file.exists():
        with open(checksum_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 2:
                    forensic_data["checksums"][parts[0]] = parts[1]
        print(f"Loaded {len(forensic_data['checksums'])} checksums")
    
    forensic_data["status"] = "success"
    return forensic_data

def analyze_memory_dump(memory_dump_path: str) -> Dict[str, Any]:
    """Analyze memory dump using Volatility framework."""
    print(f"Analyzing memory dump: {memory_dump_path}")
    
    analysis_results = {
        "memory_dump_path": memory_dump_path,
        "analysis_timestamp": datetime.now().isoformat(),
        "suspicious_processes": [],
        "network_artifacts": []
    }
    
    print("Memory dump analysis using Volatility:")
    print(f"  volatility -f {memory_dump_path} pslist")
    print(f"  volatility -f {memory_dump_path} netscan")
    
    return analysis_results

def analyze_disk_image(disk_image_path: str) -> Dict[str, Any]:
    """Analyze disk image using forensic tools."""
    print(f"Analyzing disk image: {disk_image_path}")
    
    analysis_results = {
        "disk_image_path": disk_image_path,
        "analysis_timestamp": datetime.now().isoformat(),
        "file_system_artifacts": [],
        "deleted_files": []
    }
    
    print("Disk image analysis using Autopsy/EnCase:")
    print(f"  autopsy {disk_image_path}")
    
    return analysis_results

# Example Usage:
CENTRALIZED_STORAGE = "\\\\forensics-server\\evidence"
agent_id = "SERVER-001"

forensic_evidence = retrieve_forensic_evidence(CENTRALIZED_STORAGE, agent_id)

if forensic_evidence["status"] == "success":
    print(f"Memory dump: {forensic_evidence['memory_dump']}")
    print(f"Disk image: {forensic_evidence['disk_image']}")
    
    if forensic_evidence["memory_dump"]:
        memory_analysis = analyze_memory_dump(forensic_evidence["memory_dump"])
    
    if forensic_evidence["disk_image"]:
        disk_analysis = analyze_disk_image(forensic_evidence["disk_image"])
else:
    print(f"Error: {forensic_evidence['message']}")




#### 8.2.2. Step 2: Root Cause Analysis (RCA)

**Objective:** To determine the fundamental security gap that allowed the incident to occur.

**Actions:**
- Ask the **Five Whys**: Continuously ask "Why?" to drill down from the symptom to the root cause.
- **Example RCA Path:**
  - **Why 1:** Files were encrypted. → Because WannaCry spread on the network.
  - **Why 2:** Why did WannaCry spread? → Because SMBv1 was exploited via EternalBlue.
  - **Why 3:** Why was SMBv1 exploited? → Because MS17-010 patch was not applied.
  - **Why 4:** Why was the patch not applied? → Because the system was not included in the patch management cycle.
  - **Why 5:** Why was it not included? → Because it was miscategorized as non-critical, and our patch SLA for non-critical assets is 30 days.
- Document the root cause in the RCA report.
- Identify contributing factors (e.g., lack of network segmentation, insufficient monitoring).

**Technical Steps:**
- Review patch management logs to identify which systems were missing MS17-010.
- Check network segmentation to see if lateral movement could have been prevented.
- Analyze firewall logs to see if C2 communication could have been detected.
- Review monitoring and alerting to identify gaps in detection capabilities.

#### 8.2.3. Step 3: Implement Preventative Measures (Technical Steps)

**Objective:** To apply specific technical controls to prevent this and similar incidents from happening again.

**8.2.3.1. Patch Management Automation**
- **Action:** Implement a centralized, automated patch management system (e.g., WSUS, SCCM, or a third-party tool).
- **Technical Steps:**
  - Configure the system to automatically deploy critical and security-related patches within a 72-hour SLA for all systems.
  - Use Group Policy or equivalent to enforce patch deployment.
  - Create automated reports to track patch compliance.
  - Use the Wazuh API to verify patch levels on all agents and alert on non-compliance.
  


In [ ]:
# Example: Check for a specific KB on a remote machine using Wazuh API
agent_id = "001"  # Example agent ID
command = "powershell -Command \"Get-Hotfix -Id KB4012598\""
result = wazuh_client.run_command_on_agent(agent_id, command)
if result.get('error') == 0 and result.get('data', {}).get('output'):
    print(f"Patch KB4012598 is installed on agent {agent_id}")
else:
    print(f"Patch KB4012598 is NOT installed on agent {agent_id}")


**8.2.3.2. Detection Rules Enhancement**
- **Action:** Enhance EDR/SIEM policies by implementing Sigma rules for high-fidelity detection of WannaCry indicators.
- **Technical Steps:**
  - Convert the Sigma rule below into the native format for your EDR/SIEM solution (e.g., Wazuh, Splunk, Elastic).
  - This rule identifies WannaCry ransomware by its specific process image names.
  - Deploy the rule to production monitoring systems.
  - Test the rule to ensure proper detection without excessive false positives.

**Sigma Rule for WannaCry:**


In [ ]:
title: WannaCry Ransomware Execution
id: 58a799a3-179a-4354-8a40-43a0595b1e1e
status: stable
description: Detects the execution of processes associated with the WannaCry ransomware.
references:
    - https://www.microsoft.com/en-us/security/blog/2017/05/12/wannacrypt-ransomware-attack-guidance-for-analysts-and-it-pros/

date: 2026/01/17
logsource:
    category: process_creation
    product: windows
detection:
    selection:
        Image|endswith:
            - '\tasksche.exe'
            - '\@WanaDecryptor@.exe'
            - '\wnry.exe'
            - '\wcry.exe'
    condition: selection
falsepositives:
    - Unlikely
level: critical


**Implementation in Wazuh:**
- Convert the Sigma rule to Wazuh rule format and add it to your `local_rules.xml`.
- The rule triggers on Windows event logs (e.g., Sysmon Event ID 1) where the `process.name` matches the specified executables.


**8.2.3.3. Patch Management Automation**
- **Action:** Implement a centralized, automated patch management system.
- **Technical Steps:**
  - Configure automated deployment of critical and security patches within 72-hour SLA.
  - Use Group Policy or tools like WSUS/SCCM for enforcement.
  - Monitor patch compliance using Wazuh API.


**8.2.3.4. Disable SMBv1 Organization-Wide**
- **Action:** Ensure SMBv1 is disabled on all systems.
- **Technical Steps:**
  - Use Group Policy to disable SMBv1 on all Windows systems.
  - Verify compliance using Wazuh or similar tools.


In [ ]:
import requests
from typing import Dict, Any

class WazuhAPIClient:
    """Client for interacting with the Wazuh API"""
    def __init__(self, api_url: str, username: str, password: str):
        self.api_url = api_url.rstrip('/')
        self.auth = (username, password)
        self.session = requests.Session()
        self.session.verify = False
    
    def run_command_on_agent(self, agent_id: str, command: str) -> Dict[str, Any]:
        """Run a custom command on an agent using active response."""
        data = {"command": "run-command", "agent_id": agent_id, "arguments": [command]}
        response = self.session.put(f"{self.api_url}/active-response", json=data, auth=self.auth)
        return response.json()

def verify_smbv1_disabled(wazuh_client: WazuhAPIClient, agent_id: str) -> bool:
    """
    Verify that SMBv1 is disabled on a specific agent.
    
    Args:
        wazuh_client: WazuhAPIClient instance
        agent_id: ID of the agent to verify
    
    Returns:
        True if SMBv1 is disabled, False otherwise
    """
    print(f"Verifying SMBv1 status on agent {agent_id}...")
    command = "powershell -Command \"(Get-WindowsOptionalFeature -Online -FeatureName SMB1Protocol).State\""
    result = wazuh_client.run_command_on_agent(agent_id, command)
    
    if result.get('error') == 0:
        output = result.get('data', {}).get('output', '')
        if "Disabled" in output:
            print(f"SMBv1 is DISABLED on agent {agent_id}")
            return True
        else:
            print(f"SMBv1 is NOT disabled on agent {agent_id}. Output: {output}")
            return False
    else:
        print(f"Failed to verify SMBv1 status on agent {agent_id}: {result.get('message')}")
        return False

def disable_smbv1_on_agent(wazuh_client: WazuhAPIClient, agent_id: str) -> bool:
    """
    Disable SMBv1 on a specific agent.
    
    Args:
        wazuh_client: WazuhAPIClient instance
        agent_id: ID of the agent
    
    Returns:
        True if successful, False otherwise
    """
    print(f"Disabling SMBv1 on agent {agent_id}...")
    command = "powershell -Command \"dism /online /norestart /disable-feature /featurename:SMB1Protocol\""
    result = wazuh_client.run_command_on_agent(agent_id, command)
    
    if result.get('error') == 0:
        print(f"SMBv1 disabled successfully on agent {agent_id}")
        return True
    else:
        print(f"Failed to disable SMBv1 on agent {agent_id}: {result.get('message')}")
        return False

# Example Usage:
API_URL = "https://wazuh-manager.example.com:55000"
USERNAME = "admin"
PASSWORD = "password"

wazuh_client = WazuhAPIClient(API_URL, USERNAME, PASSWORD)
agent_ids = ["001", "002", "003"]  # List of agent IDs to verify

for agent_id in agent_ids:
    is_disabled = verify_smbv1_disabled(wazuh_client, agent_id)
    if not is_disabled:
        print(f"Attempting to disable SMBv1 on agent {agent_id}...")
        disable_smbv1_on_agent(wazuh_client, agent_id)
        # Verify again after disabling
        verify_smbv1_disabled(wazuh_client, agent_id)




**8.2.3.5. Implement Application Whitelisting**
- **Action:** Use AppLocker or Windows Defender Application Control to prevent unauthorized executables.
- **Technical Steps:**
  - Create AppLocker rules to allow only approved applications.
  - Block execution from temporary directories (%TEMP%, %WINDIR%\Temp).
  - Implement audit mode first to identify legitimate applications, then enforce.



In [ ]:
import requests
from typing import Dict, Any

class WazuhAPIClient:
    """Client for interacting with the Wazuh API"""
    def __init__(self, api_url: str, username: str, password: str):
        self.api_url = api_url.rstrip('/')
        self.auth = (username, password)
        self.session = requests.Session()
        self.session.verify = False
    
    def get_alerts(self, q: str, time_range: str = "1h") -> Dict[str, Any]:
        """Get alerts based on a query"""
        params = {"q": q, "timeframe": time_range}
        response = self.session.get(f"{self.api_url}/alerts", params=params, auth=self.auth)
        return response.json()

def check_temp_directory_execution(wazuh_client: WazuhAPIClient, agent_id: str) -> Dict[str, Any]:
    """
    Check for execution from temporary directories on a specific agent.
    
    Args:
        wazuh_client: WazuhAPIClient instance
        agent_id: ID of the agent to check
    
    Returns:
        Dictionary containing alerts for temp directory execution
    """
    print(f"Checking for execution from temporary directories on agent {agent_id}...")
    query = f"rule.description:Execution from temporary directory AND agent.id:{agent_id}"
    result = wazuh_client.get_alerts(q=query)
    
    if result.get('data', {}).get('total_affected_items', 0) > 0:
        print(f"Found {result['data']['total_affected_items']} alerts for execution from temp directories on agent {agent_id}")
        return {"status": "alerts_found", "count": result['data']['total_affected_items'], "alerts": result}
    else:
        print(f"No alerts found for temp directory execution on agent {agent_id}")
        return {"status": "no_alerts", "count": 0}

# Example Usage:
API_URL = "https://wazuh-manager.example.com:55000"
USERNAME = "admin"
PASSWORD = "password"

wazuh_client = WazuhAPIClient(API_URL, USERNAME, PASSWORD)
agent_ids = ["001", "002", "003"]  # List of agent IDs to check

for agent_id in agent_ids:
    result = check_temp_directory_execution(wazuh_client, agent_id)
    if result["status"] == "alerts_found":
        print(f"Action required: Review and implement AppLocker rules on agent {agent_id}")
    else:
        print(f"Agent {agent_id} is compliant with application whitelisting")




#### 8.2.4. Step 4: Final Incident Reporting & Lessons Learned

**Objective:** To formally close the incident and share knowledge across the organization.

**Actions:**
- **Write the Final Incident Report:** Compile all findings into a single document including:
  - Executive Summary (1-2 pages for leadership).
  - Detailed Timeline of Events.
  - Root Cause Analysis.
  - Impact Assessment (systems affected, data compromised, downtime).
  - Preventative Measures Implemented.
  - Recommendations for Future Improvements.
  
- **Conduct a Lessons Learned Meeting:** Present the report to stakeholders (technical teams, management, leadership). Discuss:
  - What went well during the response.
  - What could be improved.
  - Effectiveness of the incident response plan.
  - Changes needed to prevent recurrence.
  
- **Update Documentation:** Revise this runbook, network diagrams, security policies, and playbooks to reflect:
  - New preventative controls.
  - Lessons learned from the incident.
  - Updated contact information and escalation procedures.
  - New detection signatures and rules.
  
- **Assign Ownership and Timelines:** For each preventative measure, assign an owner and a completion deadline. Track progress until all measures are implemented.

### 8.3. Automated Reporting Script

This script generates a summary report of the automated actions taken during the incident.



In [ ]:
import json
from datetime import datetime
from typing import Dict, List, Any

def generate_incident_report(response_log: List[Dict], affected_hosts: List[str], incident_id: str, filename: str) -> Dict[str, Any]:
    """
    Generate and save a JSON report of the incident response actions.
    """
    print("Generating incident report...")
    
    report = {
        "incident_id": incident_id,
        "start_time": response_log[0]['timestamp'] if response_log else 'N/A',
        "end_time": datetime.now().isoformat(),
        "affected_hosts": affected_hosts,
        "total_affected": len(affected_hosts),
        "action_log": response_log,
        "total_actions": len(response_log),
        "successful_actions": sum(1 for action in response_log if action.get('status') == 'SUCCESS'),
        "failed_actions": sum(1 for action in response_log if action.get('status') == 'FAILED')
    }
    
    with open(filename, 'w') as f:
        json.dump(report, f, indent=4)
    
    print(f"Incident report saved to {filename}")
    return report

def generate_executive_summary(incident_report: Dict[str, Any]) -> str:
    """
    Generate an executive summary from the incident report.
    """
    summary = f"""
INCIDENT RESPONSE EXECUTIVE SUMMARY
====================================

Incident ID: {incident_report['incident_id']}
Start Time: {incident_report['start_time']}
End Time: {incident_report['end_time']}

IMPACT:
- Total Affected Hosts: {incident_report['total_affected']}
- Affected Systems: {', '.join(incident_report['affected_hosts'])}

RESPONSE ACTIONS:
- Total Actions Taken: {incident_report['total_actions']}
- Successful Actions: {incident_report['successful_actions']}
- Failed Actions: {incident_report['failed_actions']}

STATUS: {'SUCCESSFUL' if incident_report['failed_actions'] == 0 else 'COMPLETED WITH ISSUES'}
"""
    return summary

# Example Usage:
# response_log = [
#     {"timestamp": "2026-01-17T10:00:00", "action": "THREAT_HUNT", "status": "SUCCESS"},
#     {"timestamp": "2026-01-17T10:05:00", "action": "ISOLATION", "status": "SUCCESS"},
#     ...
# ]
# affected_hosts = ["SERVER-001", "SERVER-002"]
# incident_id = "WC-20260117-100000"
# report = generate_incident_report(response_log, affected_hosts, incident_id, "incident_report.json")
# summary = generate_executive_summary(report)
# print(summary)




---


## 9. Appendix A: End-to-End Incident Checklist {#checklist}

This comprehensive checklist covers every critical step from initial preparation to post-incident closure. Use it to ensure all actions are completed and to track progress during an incident.

### 9.1. Phase 1: Preparation & Prevention

#### 9.1.1. System Hardening
- [ ] MS17-010 patch is deployed on all Windows systems.
- [ ] Deployment verified through patch management system or Wazuh.
- [ ] SMBv1 protocol is disabled via Group Policy on all systems.
- [ ] Compliance verified through Group Policy reports or Wazuh.
- [ ] Windows Firewall is enabled with strict inbound rules.
- [ ] Firewall rules are documented and reviewed.
- [ ] Unnecessary services (especially RDP) are disabled where not needed.
- [ ] Service configuration is documented.
- [ ] Application whitelisting (AppLocker/WDAC) is implemented.
- [ ] Whitelisting rules are tested and documented.

#### 9.1.2. Network Security
- [ ] Port 445 is blocked from external access at the perimeter firewall.
- [ ] Firewall rule is documented with business justification.
- [ ] Network segmentation is implemented (separate VLANs for users, servers, critical systems).
- [ ] VLAN configuration is documented.
- [ ] IDS/IPS is deployed and configured.
- [ ] IDS/IPS rules for EternalBlue are enabled.
- [ ] Network monitoring (NetFlow/sFlow) is configured.
- [ ] Monitoring is validated to capture traffic.

#### 9.1.3. Backup Strategy
- [ ] 3-2-1 backup rule is implemented (3 copies, 2 media types, 1 offsite).
- [ ] Backup configuration is documented.
- [ ] Backups are disconnected from the network (air-gapped).
- [ ] Backup storage is verified to be offline or immutable.
- [ ] Backup restoration procedure is tested periodically.
- [ ] Last restoration test is documented with date and results.
- [ ] RTO and RPO targets are defined and met.
- [ ] Targets are documented and communicated to stakeholders.

#### 9.1.4. Monitoring & Detection
- [ ] Windows Event Logging is enabled on all systems.
- [ ] Log retention is configured for a defined period.
- [ ] Sysmon is deployed on all systems.
- [ ] Sysmon rules are configured for process monitoring.
- [ ] File Integrity Monitoring (FIM) is enabled for critical files.
- [ ] FIM is configured for system directories and critical applications.
- [ ] Network flow monitoring (NetFlow/sFlow) is configured.
- [ ] Flow data is being collected and retained.
- [ ] Threat hunting system is deployed.
- [ ] Threat hunting system is tested and operational.

#### 9.1.5. Incident Response Team
- [ ] Incident Response team is formally established.
- [ ] All team members are identified and trained.
- [ ] Contact information is current and documented.
- [ ] Escalation procedures are defined and communicated.
- [ ] On-call rotation is established.
- [ ] War room procedures are documented.

### 9.2. Phase 2: Detection & Analysis

#### 9.2.1. Alert Verification
- [ ] Alert is received and logged with timestamp.
- [ ] Alert source is verified as trusted.
- [ ] Alert is checked against multiple detection sources (SIEM, EDR, IDS).
- [ ] IOC presence is confirmed on suspected system.
- [ ] False positive is ruled out.
- [ ] Incident Response team is activated.

#### 9.2.2. Initial Assessment
- [ ] Incident Commander is assigned.
- [ ] War room is established (physical or virtual).
- [ ] All affected systems are identified.
- [ ] Infection timeline is estimated.
- [ ] Number of impacted hosts is documented.
- [ ] Critical systems at risk are identified.
- [ ] Executive leadership is notified.

#### 9.2.3. Threat Hunt
- [ ] Automated threat hunting is executed.
- [ ] All systems are scanned for WannaCry IOCs.
- [ ] Additional infected systems are identified.
- [ ] All findings are documented.
- [ ] Threat hunt results are reviewed and verified.

### 9.3. Phase 3: Containment

#### 9.3.1. Host Isolation
- [ ] All confirmed infected hosts are isolated from the network.
- [ ] Isolation method is documented (cable disconnect, NIC disable, etc.).
- [ ] Hosts are NOT shut down (to preserve volatile memory).
- [ ] Isolation is verified through network connectivity tests.

#### 9.3.2. Network Blocking
- [ ] SMB port 445 is blocked across all network segments.
- [ ] Firewall rule is implemented and verified.
- [ ] RDP port 3389 is blocked from affected subnets.
- [ ] Firewall rule is implemented and verified.
- [ ] Known Tor exit nodes are blocked.
- [ ] Firewall rule is implemented and verified.
- [ ] Known C2 domains are blocked.
- [ ] Firewall rule is implemented and verified.

#### 9.3.3. System-Level Containment
- [ ] Malware processes are killed on all infected systems.
- [ ] Command execution is verified through Wazuh or similar.
- [ ] Malicious service (mssecsvc2.0) is disabled.
- [ ] Service status is verified.
- [ ] Startup entries are removed.
- [ ] Registry is verified to be clean.

#### 9.3.4. Evidence Preservation
- [ ] Memory dumps are collected from infected systems.
- [ ] Disk images are created from infected systems.
- [ ] Chain of custody is maintained.
- [ ] Evidence is stored in secure location.
- [ ] Access to evidence is restricted.

#### 9.3.5. Communication
- [ ] All department heads are notified.
- [ ] IT staff are informed of containment actions.
- [ ] Executive leadership is briefed.
- [ ] Hourly status updates are prepared.
- [ ] External resources are engaged if needed.

### 9.4. Phase 4: Eradication

#### 9.4.1. Forensic Collection
- [ ] Memory dumps are collected before remediation.
- [ ] Disk images are created before remediation.
- [ ] Chain of custody is maintained.
- [ ] Evidence is securely stored.

#### 9.4.2. Malware Removal
- [ ] Automated malware removal scripts are executed.
- [ ] All known WannaCry files are deleted.
- [ ] Registry entries are removed.
- [ ] Temporary files are cleared.
- [ ] Removal is verified through IOC scanning.

#### 9.4.3. Patching & Hardening
- [ ] MS17-010 patch is deployed to all systems.
- [ ] Patch deployment is verified.
- [ ] SMBv1 is disabled on all systems.
- [ ] Compliance is verified through Group Policy reports.
- [ ] Firewall is enabled and rules are in place.
- [ ] Firewall configuration is verified.
- [ ] System hardening configurations are verified.

#### 9.4.4. Verification
- [ ] All systems are scanned for remaining IOCs.
- [ ] No malware indicators are found.
- [ ] Registry is verified to be clean.
- [ ] Network logs show no C2 communication attempts.

### 9.5. Phase 5: Recovery

#### 9.5.1. Backup Verification
- [ ] Backup integrity is verified.
- [ ] Backup point before infection is identified.
- [ ] Backup restoration is tested on isolated system.
- [ ] Data integrity is verified after restoration.

#### 9.5.2. System Restoration
- [ ] Systems are restored from known-good backups.
- [ ] Restoration is completed for all affected systems.
- [ ] Restored systems are verified to be clean.
- [ ] Critical business functions are tested.

#### 9.5.3. System Verification
- [ ] Full antivirus scan is performed on restored systems.
- [ ] No malware IOCs are detected.
- [ ] All security patches are installed.
- [ ] Firewall rules are in place.
- [ ] Network connectivity is verified.
- [ ] Access to critical services is verified.

#### 9.5.4. Phased Reconnection
- [ ] Non-critical systems are restored first.
- [ ] Systems are monitored for reinfection.
- [ ] Business-critical systems are restored.
- [ ] Systems are monitored for reinfection.
- [ ] End-user systems are restored.
- [ ] Systems are monitored for reinfection.
- [ ] Full operations are resumed.
- [ ] Continuous monitoring is maintained.

### 9.6. Phase 6: Post-Incident & Continuous Improvement

#### 9.6.1. Forensic Analysis
- [ ] Forensic images are analyzed using Volatility, Autopsy, or EnCase.
- [ ] Attack timeline is reconstructed.
- [ ] All compromised systems and accounts are identified.
- [ ] Data exfiltration is assessed.
- [ ] Forensic analysis report is completed.

#### 9.6.2. Root Cause Analysis
- [ ] Five Whys analysis is completed.
- [ ] Root cause is identified and documented.
- [ ] Contributing factors are identified.
- [ ] RCA report is completed.

#### 9.6.3. Preventative Measures
- [ ] Patch management automation is implemented.
  - [ ] Centralized patch management system is deployed.
  - [ ] 72-hour SLA for critical patches is enforced.
  - [ ] Automated compliance reporting is configured.
  - [ ] Wazuh API is configured to verify patch levels.
- [ ] Network segmentation is enhanced.
  - [ ] Micro-segmentation rules are implemented.
  - [ ] East-west traffic is restricted.
  - [ ] SMBv1 traffic is blocked at network level.
- [ ] EDR tuning is completed.
  - [ ] Custom detection rules are created.
  - [ ] Automated response actions are configured.
  - [ ] Rules are tested and validated.
- [ ] Backup system auditing is implemented.
  - [ ] Immutability features are enabled.
  - [ ] Periodic restoration tests are scheduled.
  - [ ] RTO/RPO targets are verified.
- [ ] SMBv1 is disabled organization-wide.
  - [ ] Group Policy is deployed.
  - [ ] Compliance is verified.
- [ ] Application whitelisting is implemented.
  - [ ] AppLocker/WDAC rules are created.
  - [ ] Rules are tested and enforced.

#### 9.6.4. Final Reporting
- [ ] Final incident report is written.
- [ ] Report includes executive summary, timeline, RCA, impact, and recommendations.
- [ ] Report is reviewed and approved.
- [ ] Lessons learned meeting is scheduled.
- [ ] All stakeholders are invited to the meeting.
- [ ] Meeting is conducted and findings are discussed.
- [ ] Action items are assigned with owners and deadlines.

#### 9.6.5. Documentation Updates
- [ ] This runbook is updated with lessons learned.
- [ ] Network diagrams are updated.
- [ ] Security policies are updated.
- [ ] Playbooks are updated.
- [ ] Detection signatures and rules are updated.
- [ ] Contact information is updated.

#### 9.6.6. Closure
- [ ] All action items are tracked to completion.
- [ ] Preventative measures are verified to be effective.
- [ ] Incident is formally closed.
- [ ] Closure is documented and communicated.

---


## 10. Appendix B: Communication Templates {#communication}

### 10.1. Initial Alert Notification

**SUBJECT: CRITICAL - WannaCry Ransomware Detected**

**TO:** Executive Leadership, Department Heads, IT Security Team

**PRIORITY:** CRITICAL  
**TIMESTAMP:** {timestamp}  
**INCIDENT ID:** {incident_id}

---

**SITUATION:**  
WannaCry ransomware has been detected on {affected_count} system(s) in our infrastructure. The incident response team has been activated and is taking immediate action to contain the threat.

**DETECTION METHOD:**  
- {detection_method}  
- Confidence Level: {confidence}%

**IMMEDIATE ACTIONS TAKEN:**  
1. Incident response team activated
2. Affected systems isolated from network
3. Malware processes terminated
4. Network-level containment implemented
5. Threat hunting initiated

**AFFECTED SYSTEMS:**  
{affected_systems}

**NEXT STEPS:**  
- Detailed forensic analysis underway
- Hourly status updates will be provided
- Recovery procedures being prepared
- External resources engaged if necessary

**POINT OF CONTACT:**  
Incident Commander: {ic_name} ({ic_phone})  
Security Operations: {soc_contact}

**STATUS:** ONGOING  
**NEXT UPDATE:** In 1 hour

---

### 10.2. Hourly Status Update

**SUBJECT:** WannaCry Incident Status Update - {hours_elapsed} Hours

**INCIDENT ID:** {incident_id}  
**TIMESTAMP:** {timestamp}  
**STATUS:** {current_status}

---

**SITUATION UPDATE:**  
- Initial infected systems: {initial_count}
- Additional systems identified: {additional_count}
- Total affected: {total_count}
- New infections detected: {new_infections}

**CONTAINMENT STATUS:**  
- Network isolation: COMPLETE
- Malware processes: TERMINATED
- C2 traffic: BLOCKED
- Lateral movement: PREVENTED

**ERADICATION PROGRESS:**  
- Forensic analysis: {forensic_percent}% complete
- Malware removal: {removal_percent}% complete
- System patching: {patching_percent}% complete

**RECOVERY TIMELINE:**  
- Backup verification: {backup_status}
- Data restoration: {restoration_status}
- Estimated completion: {completion_time}

**BUSINESS IMPACT:**  
- Systems down: {systems_down}
- Users affected: {users_affected}
- Critical services: {critical_services_status}

**NEXT ACTIONS:**  
1. Continue forensic analysis
2. Begin system restoration
3. Monitor for new infections
4. Prepare recovery procedures

**POINT OF CONTACT:**  
Incident Commander: {ic_name}

**NEXT UPDATE:** In 1 hour

---

### 10.3. Final Incident Closure Notification

**SUBJECT:** WannaCry Incident - CLOSED

**INCIDENT ID:** {incident_id}  
**TIMESTAMP:** {timestamp}

---

**SITUATION:**  
The WannaCry incident has been successfully contained, eradicated, and recovered. All affected systems have been restored and verified to be clean.

**INCIDENT SUMMARY:**  
- Total affected systems: {total_affected}
- Duration: {duration}
- Root cause: {root_cause}
- Data recovered: {data_recovered}%

**ACTIONS TAKEN:**  
- All systems isolated and cleaned
- All vulnerabilities patched
- All data restored from backups
- All preventative measures implemented

**PREVENTATIVE MEASURES IMPLEMENTED:**  
1. {measure_1}
2. {measure_2}
3. {measure_3}

**LESSONS LEARNED:**  
- {lesson_1}
- {lesson_2}
- {lesson_3}

**NEXT STEPS:**  
- Lessons learned meeting scheduled for {date}
- Runbook and procedures will be updated
- Security awareness training will be conducted

**POINT OF CONTACT:**  
Incident Commander: {ic_name}

---


## 11. Appendix C: Tools & Resources {#tools}

### 11.1. Threat Hunting & Detection
- **Yara:** Malware pattern matching and detection
- **VirusTotal:** File hash lookup and analysis
- **MISP:** Malware Information Sharing Platform
- **AlienVault OTX:** Open Threat Exchange for threat intelligence

### 11.2. Forensics & Analysis
- **Volatility:** Memory forensics framework for analyzing memory dumps
- **Autopsy:** Digital forensics platform for disk analysis
- **EnCase:** Commercial forensic investigation platform
- **FTK Imager:** Forensic imaging and analysis tool

### 11.3. Incident Response & Monitoring
- **Splunk:** Security Information and Event Management (SIEM)
- **Wazuh:** Threat detection and response platform
- **CrowdStrike:** Endpoint Detection and Response (EDR)
- **Zeek:** Network monitoring and analysis platform

### 11.4. Malware Analysis
- **Cuckoo Sandbox:** Automated malware analysis system
- **IDA Pro:** Disassembler and debugger for reverse engineering
- **Ghidra:** Reverse engineering framework (open-source)
- **Wireshark:** Network protocol analyzer

### 11.5. Decryption & Recovery
- **No-More-Ransom.org:** Free decryption tools for various ransomware families
- **Kaspersky Decryption Tools:** Ransomware decryption utilities
- **Law Enforcement Resources:** FBI, Interpol, CISA

### 11.6. External Resources
- **CISA:** Cybersecurity and Infrastructure Security Agency (alerts and advisories)
- **FBI:** Federal Bureau of Investigation (incident reporting and support)
- **Interpol:** International Criminal Police Organization
- **Microsoft Security:** Security bulletins and patches

---


## 12. Appendix D: Decision Trees {#decision-trees}

### 12.1. Detection Decision Tree

```
START: Alert Received
|
+---> Verify Alert Source
|     |
|     +---> Known trusted source?
|     |     |
|     |     +---> YES: Proceed to analysis
|     |     +---> NO: Cross-reference with other sources
|     |           |
|     |           +---> Multiple confirmations?
|     |                 |
|     |                 +---> YES: Proceed to analysis
|     |                 +---> NO: False positive - monitor
|
+---> Analyze Alert Content
|     |
|     +---> Contains WannaCry IOCs?
|     |     |
|     |     +---> YES: High confidence - activate response
|     |     +---> NO: Check for behavioral indicators
|     |           |
|     |           +---> File encryption observed?
|     |           |     |
|     |           |     +---> YES: Medium confidence - investigate
|     |           |     +---> NO: Low confidence - monitor
|     |           |
|     |           +---> Network scanning detected?
|     |           |     |
|     |           |     +---> YES: Medium confidence - investigate
|     |           +---> NO: Insufficient evidence - monitor
|
+---> Determine Response Level
      |
      +---> High confidence: Immediate containment
      +---> Medium confidence: Rapid investigation
      +---> Low confidence: Enhanced monitoring
```

### 12.2. Containment Decision Tree

```
START: WannaCry Confirmed
|
+---> Assess Infection Scope
|     |
|     +---> Single system?
|     |     |
|     |     +---> YES: Isolate system immediately
|     |     +---> NO: Multiple systems affected?
|     |           |
|     |           +---> YES: Network segmentation required
|     |           +---> NO: Check for lateral movement
|
+---> Evaluate Containment Options
|     |
|     +---> Can isolate at host level?
|     |     |
|     |     +---> YES: Disconnect network cable/disable NIC
|     |     +---> NO: Network-level controls available?
|     |           |
|     |           +---> YES: Firewall rules/VLAN isolation
|     |           +---> NO: Domain isolation/shutdown
|
+---> Implement Containment
|     |
|     +---> Block SMB ports (445, 139)
|     +---> Block Tor exit nodes
|     +---> Block known C2 domains
|     +---> Kill malware processes
|     +---> Preserve evidence
|
+---> Verify Containment
      |
      +---> No new infections detected?
      +---> Lateral movement stopped?
      +---> Evidence preserved?
```

### 12.3. Recovery Decision Tree

```
START: Eradication Complete
|
+---> Assess Recovery Options
|     |
|     +---> Are backups available?
|     |     |
|     |     +---> YES: Check backup integrity
|     |     |     |
|     |     |     +---> Backup valid?
|     |     |           |
|     |     |           +---> YES: Restore from backup
|     |     |           |     |
|     |     |           |     +---> Test restoration
|     |     |           |     +---> Verify data integrity
|     |     |           |     +---> Reconnect to network
|     |     |           |
|     |     |           +---> NO: Backup corrupted
|     |     |                 |
|     |     |                 +---> Try alternative backup
|     |     |                 +---> If none available: Rebuild
|     |     |
|     |     +---> NO: No backups available
|     |           |
|     |           +---> Is decryption available?
|     |                 |
|     |                 +---> YES: Attempt decryption
|     |                 |     |
|     |                 |     +---> Use No-More-Ransom tools
|     |                 |     +---> Verify decryption success
|     |                 |
|     |                 +---> NO: Rebuild system
|     |                       |
|     |                       +---> Clean OS installation
|     |                       +---> Restore from oldest backup
|     |                       +---> Accept data loss
|     |
|     +---> Prioritize Recovery
|           |
|           +---> Phase 1: Non-critical systems
|           +---> Phase 2: Business-critical systems
|           +---> Phase 3: End-user systems
|           +---> Phase 4: Full operations
|
+---> Verify System Security
|     |
|     +---> Patches applied?
|     +---> Malware removed?
|     +---> Firewall rules in place?
|     +---> Monitoring enabled?
|
+---> Recovery Complete
      |
      +---> Begin post-incident activities
```

---


## 13. Appendix E: Comprehensive Automated Incident Response Script {#automation-scripts}

### 13.1. Complete End-to-End WannaCry Incident Response Automation

This section contains a single, comprehensive Python script that automates the entire WannaCry incident response workflow from detection through post-incident activities. This script orchestrates all phases of the incident response lifecycle.



In [ ]:
import json
import requests
import os
from datetime import datetime
from typing import Dict, List, Any
from pathlib import Path

class WazuhAPIClient:
    """Client for interacting with the Wazuh API to execute commands on agents."""
    def __init__(self, api_url: str, username: str, password: str):
        self.api_url = api_url.rstrip('/')
        self.auth = (username, password)
        self.session = requests.Session()
        self.session.verify = False
    
    def get_agents(self) -> Dict[str, Any]:
        """Get list of all agents"""
        response = self.session.get(f"{self.api_url}/agents", auth=self.auth)
        return response.json()
    
    def get_agent(self, agent_id: str) -> Dict[str, Any]:
        """Get information about a specific agent"""
        response = self.session.get(f"{self.api_url}/agents/{agent_id}", auth=self.auth)
        return response.json()
    
    def get_alerts(self, q: str, time_range: str = "1h") -> Dict[str, Any]:
        """Get alerts based on a query"""
        params = {"q": q, "timeframe": time_range}
        response = self.session.get(f"{self.api_url}/alerts", params=params, auth=self.auth)
        return response.json()

    def run_active_response(self, command: str, agent_id: str, arguments: list = None) -> Dict[str, Any]:
        """Run an active response command on an agent."""
        data = {"command": command, "agent_id": agent_id}
        if arguments:
            data["arguments"] = arguments
        response = self.session.put(f"{self.api_url}/active-response", json=data, auth=self.auth)
        return response.json()
    
    def run_command_on_agent(self, agent_id: str, command: str) -> Dict[str, Any]:
        """Run a custom command on an agent using active response."""
        return self.run_active_response("run-command", agent_id, [command])

class CrowdStrikeForensicsClient:
    """Client for interacting with CrowdStrike Falcon API for forensic collection."""
    def __init__(self, client_id: str, client_secret: str, base_url: str = "https://api.crowdstrike.com"):
        self.client_id = client_id
        self.client_secret = client_secret
        self.base_url = base_url.rstrip('/')
        self.session = requests.Session()
        self.access_token = None
        self._authenticate()
    
    def _authenticate(self) -> bool:
        """Authenticate with CrowdStrike API and obtain access token."""
        auth_url = f"{self.base_url}/oauth2/token"
        auth_data = {"client_id": self.client_id, "client_secret": self.client_secret}
        try:
            response = self.session.post(auth_url, data=auth_data)
            response.raise_for_status()
            token_response = response.json()
            self.access_token = token_response.get('access_token')
            self.session.headers.update({"Authorization": f"Bearer {self.access_token}"})
            return True
        except Exception as e:
            print(f"CrowdStrike authentication failed: {e}")
            return False
    
    def get_device_by_hostname(self, hostname: str) -> Dict[str, Any]:
        """Get device ID by hostname."""
        search_url = f"{self.base_url}/devices/queries/devices/v1"
        params = {"filter": f"hostname:'{hostname}'"}
        try:
            response = self.session.get(search_url, params=params)
            response.raise_for_status()
            data = response.json()
            if data.get('resources'):
                return {"device_id": data['resources'][0], "status": "success"}
            else:
                return {"status": "error", "message": f"Device {hostname} not found"}
        except Exception as e:
            return {"status": "error", "message": str(e)}
    
    def initiate_live_response_session(self, device_id: str) -> Dict[str, Any]:
        """Initiate a live response session with a device."""
        session_url = f"{self.base_url}/loggingapi/combined/sessions/v1"
        session_data = {"device_id": device_id, "timeout": 600}
        try:
            response = self.session.post(session_url, json=session_data)
            response.raise_for_status()
            data = response.json()
            session_id = data.get('resources', [{}])[0].get('session_id')
            return {"session_id": session_id, "status": "success"}
        except Exception as e:
            return {"status": "error", "message": str(e)}
    
    def execute_command(self, session_id: str, command: str) -> Dict[str, Any]:
        """Execute a command in a live response session."""
        command_url = f"{self.base_url}/loggingapi/combined/sessions/command/v1"
        command_data = {"session_id": session_id, "command": command}
        try:
            response = self.session.post(command_url, json=command_data)
            response.raise_for_status()
            return response.json()
        except Exception as e:
            return {"status": "error", "message": str(e)}

class ComprehensiveWannaCryResponse:
    """
    Comprehensive automated response system for WannaCry incidents.
    Orchestrates all phases: Detection → Containment → Eradication → Recovery → Post-Incident.
    """
    def __init__(self, wazuh_client: WazuhAPIClient, cs_client: CrowdStrikeForensicsClient = None):
        self.wazuh_client = wazuh_client
        self.cs_client = cs_client
        self.incident_id = f"WC-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
        self.affected_hosts = []
        self.response_log = []
        self.status = "INITIALIZED"
        self.evidence_location = None
    
    def log_action(self, phase: str, action: str, status: str, details: str = ""):
        """Log an automated response action."""
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "phase": phase,
            "action": action,
            "status": status,
            "details": details
        }
        self.response_log.append(log_entry)
        print(f"[{phase}] {action}: {status}")
        if details:
            print(f"  → {details}")
    
    # ===== PHASE 2: DETECTION & ANALYSIS =====
    def detect_wannacry_indicators(self) -> List[str]:
        """Phase 2: Detect WannaCry indicators across all agents."""
        print("\n=== PHASE 2: DETECTION & ANALYSIS ===")
        self.status = "DETECTING"
        
        # Query for WannaCry IOCs
        ioc_query = "rule.description:WannaCry OR rule.description:EternalBlue OR rule.description:tasksche"
        result = self.wazuh_client.get_alerts(q=ioc_query, time_range="1h")
        
        if result.get('data', {}).get('total_affected_items', 0) > 0:
            alerts = result.get('data', {}).get('affected_items', [])
            for alert in alerts:
                agent_id = alert.get('agent', {}).get('id')
                if agent_id not in self.affected_hosts:
                    self.affected_hosts.append(agent_id)
                    self.log_action("DETECTION", "IOC_FOUND", "SUCCESS", f"WannaCry detected on agent {agent_id}")
        
        print(f"Affected hosts identified: {len(self.affected_hosts)}")
        return self.affected_hosts
    
    # ===== PHASE 3: CONTAINMENT =====
    def isolate_infected_hosts(self) -> bool:
        """Phase 3: Isolate infected hosts from the network."""
        print("\n=== PHASE 3: CONTAINMENT ===")
        self.status = "CONTAINING"
        
        success_count = 0
        for agent_id in self.affected_hosts:
            # Block SMB traffic
            block_command = "powershell -Command \"netsh advfirewall firewall add rule name=\'Block SMB\' dir=in action=block protocol=tcp localport=445\""
            result = self.wazuh_client.run_command_on_agent(agent_id, block_command)
            
            if result.get('error') == 0:
                success_count += 1
                self.log_action("CONTAINMENT", "NETWORK_ISOLATION", "SUCCESS", f"SMB traffic blocked on agent {agent_id}")
            else:
                self.log_action("CONTAINMENT", "NETWORK_ISOLATION", "FAILED", f"Failed to block SMB on agent {agent_id}")
        
        print(f"Isolation complete: {success_count}/{len(self.affected_hosts)} hosts isolated")
        return success_count == len(self.affected_hosts)
    
    def block_c2_communication(self) -> bool:
        """Phase 3: Block C2 communication."""
        print("Blocking C2 communication...")
        
        c2_ips = ["195.154.35.145", "195.154.35.146"]  # Known WannaCry C2 servers
        success_count = 0
        
        for agent_id in self.affected_hosts:
            for c2_ip in c2_ips:
                block_command = f"powershell -Command \"netsh advfirewall firewall add rule name=\'Block C2 {c2_ip}\' dir=out action=block remoteip={c2_ip}\""
                result = self.wazuh_client.run_command_on_agent(agent_id, block_command)
                
                if result.get('error') == 0:
                    success_count += 1
                    self.log_action("CONTAINMENT", "C2_BLOCKING", "SUCCESS", f"C2 {c2_ip} blocked on agent {agent_id}")
        
        return success_count > 0
    
    # ===== PHASE 4: ERADICATION =====
    def collect_forensic_evidence(self) -> bool:
        """Phase 4: Collect forensic evidence using CrowdStrike."""
        print("\n=== PHASE 4: ERADICATION (Forensics) ===")
        self.status = "COLLECTING_FORENSICS"
        
        if not self.cs_client:
            self.log_action("ERADICATION", "FORENSIC_COLLECTION", "SKIPPED", "CrowdStrike client not configured")
            return False
        
        for agent_id in self.affected_hosts:
            # Get hostname from agent
            agent_info = self.wazuh_client.get_agent(agent_id)
            hostname = agent_info.get('data', {}).get('name', agent_id)
            
            # Get device ID from CrowdStrike
            device_info = self.cs_client.get_device_by_hostname(hostname)
            if device_info.get('status') != 'success':
                self.log_action("ERADICATION", "FORENSIC_COLLECTION", "FAILED", f"Could not find device {hostname} in CrowdStrike")
                continue
            
            device_id = device_info['device_id']
            
            # Initiate live response session
            session_info = self.cs_client.initiate_live_response_session(device_id)
            if session_info.get('status') != 'success':
                self.log_action("ERADICATION", "FORENSIC_COLLECTION", "FAILED", f"Could not initiate RTR session for {hostname}")
                continue
            
            session_id = session_info['session_id']
            
            # Collect volatile data
            commands = [
                "Get-Process | Select-Object Name, Id, Path",
                "netstat -ano",
                "Get-ChildItem -Path C:\\ -Recurse -Include *tasksche*,*qeriuwjhrf*,*WanaDecryptor* -ErrorAction SilentlyContinue"
            ]
            
            for cmd in commands:
                result = self.cs_client.execute_command(session_id, cmd)
                if result.get('status') == 'success':
                    self.log_action("ERADICATION", "FORENSIC_COLLECTION", "SUCCESS", f"Collected data from {hostname}")
            
            self.evidence_location = f"crowdstrike://forensics/{device_id}/{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
        return True
    
    def remove_malware(self) -> bool:
        """Phase 4: Remove WannaCry malware."""
        print("Removing malware...")
        self.status = "REMOVING_MALWARE"
        
        success_count = 0
        for agent_id in self.affected_hosts:
            commands = [
                "powershell -Command \"Remove-Item -Path 'C:\\Windows\\tasksche.exe' -Force -ErrorAction SilentlyContinue\"",
                "powershell -Command \"Remove-Item -Path 'C:\\Windows\\qeriuwjhrf' -Force -ErrorAction SilentlyContinue\"",
                "powershell -Command \"Remove-Item -Path 'HKLM:\\SYSTEM\\CurrentControlSet\\Services\\mssecsvc2.0' -Force -ErrorAction SilentlyContinue\"",
                "powershell -Command \"Get-Process | Where-Object {$_.Name -match 'tasksche|qeriuwjhrf'} | Stop-Process -Force -ErrorAction SilentlyContinue\""
            ]
            
            for command in commands:
                result = self.wazuh_client.run_command_on_agent(agent_id, command)
                if result.get('error') == 0:
                    success_count += 1
                    self.log_action("ERADICATION", "MALWARE_REMOVAL", "SUCCESS", f"Removed malware component from agent {agent_id}")
        
        return success_count > 0
    
    def apply_patches(self) -> bool:
        """Phase 4: Apply security patches."""
        print("Applying security patches...")
        
        success_count = 0
        for agent_id in self.affected_hosts:
            # Check for MS17-010 patch
            check_command = "powershell -Command \"Get-Hotfix -Id KB4012598\""
            result = self.wazuh_client.run_command_on_agent(agent_id, check_command)
            
            if result.get('error') == 0:
                self.log_action("ERADICATION", "PATCH_VERIFICATION", "SUCCESS", f"MS17-010 patch verified on agent {agent_id}")
                success_count += 1
            else:
                self.log_action("ERADICATION", "PATCH_VERIFICATION", "FAILED", f"MS17-010 patch NOT found on agent {agent_id}")
            
            # Disable SMBv1
            disable_command = "powershell -Command \"dism /online /norestart /disable-feature /featurename:SMB1Protocol\""
            result = self.wazuh_client.run_command_on_agent(agent_id, disable_command)
            
            if result.get('error') == 0:
                self.log_action("ERADICATION", "SMBV1_DISABLE", "SUCCESS", f"SMBv1 disabled on agent {agent_id}")
        
        return success_count > 0
    
    # ===== PHASE 5: RECOVERY =====
    def restore_systems(self) -> bool:
        """Phase 5: Restore systems from backups."""
        print("\n=== PHASE 5: RECOVERY ===")
        self.status = "RESTORING"
        
        for agent_id in self.affected_hosts:
            # Verify backup exists
            verify_command = "powershell -Command \"Test-Path 'E:\\Backups\\latest'\"" 
            result = self.wazuh_client.run_command_on_agent(agent_id, verify_command)
            
            if result.get('error') == 0:
                self.log_action("RECOVERY", "BACKUP_VERIFICATION", "SUCCESS", f"Backup verified for agent {agent_id}")
                
                # Restore from backup
                restore_command = "powershell -Command \"Restore-Item -Path 'E:\\Backups\\latest' -Destination 'C:\\' -Force\""
                result = self.wazuh_client.run_command_on_agent(agent_id, restore_command)
                
                if result.get('error') == 0:
                    self.log_action("RECOVERY", "SYSTEM_RESTORATION", "SUCCESS", f"System restored from backup on agent {agent_id}")
            else:
                self.log_action("RECOVERY", "BACKUP_VERIFICATION", "FAILED", f"No backup found for agent {agent_id}")
        
        return True
    
    # ===== PHASE 6: POST-INCIDENT =====
    def generate_incident_report(self, output_file: str = "incident_report.json") -> Dict[str, Any]:
        """Phase 6: Generate comprehensive incident report."""
        print("\n=== PHASE 6: POST-INCIDENT ACTIVITIES ===")
        self.status = "REPORTING"
        
        report = {
            "incident_id": self.incident_id,
            "start_time": self.response_log[0]['timestamp'] if self.response_log else 'N/A',
            "end_time": datetime.now().isoformat(),
            "affected_hosts": self.affected_hosts,
            "total_affected": len(self.affected_hosts),
            "action_log": self.response_log,
            "total_actions": len(self.response_log),
            "successful_actions": sum(1 for action in self.response_log if action.get('status') == 'SUCCESS'),
            "failed_actions": sum(1 for action in self.response_log if action.get('status') == 'FAILED'),
            "evidence_location": self.evidence_location,
            "status": "COMPLETED"
        }
        
        with open(output_file, 'w') as f:
            json.dump(report, f, indent=4)
        
        self.log_action("POST-INCIDENT", "REPORT_GENERATION", "SUCCESS", f"Report saved to {output_file}")
        return report
    
    def execute_full_response(self) -> Dict[str, Any]:
        """Execute the complete incident response workflow."""
        print("\n" + "="*60)
        print("WANNACRY COMPREHENSIVE AUTOMATED INCIDENT RESPONSE")
        print(f"Incident ID: {self.incident_id}")
        print("="*60)
        
        try:
            # Phase 2: Detection
            self.detect_wannacry_indicators()
            
            if not self.affected_hosts:
                print("No WannaCry indicators detected. Exiting.")
                return {"status": "NO_THREATS_DETECTED"}
            
            # Phase 3: Containment
            self.isolate_infected_hosts()
            self.block_c2_communication()
            
            # Phase 4: Eradication
            self.collect_forensic_evidence()
            self.remove_malware()
            self.apply_patches()
            
            # Phase 5: Recovery
            self.restore_systems()
            
            # Phase 6: Post-Incident
            report = self.generate_incident_report()
            
            self.status = "COMPLETED"
            print("\n" + "="*60)
            print("INCIDENT RESPONSE COMPLETED SUCCESSFULLY")
            print("="*60)
            
            return report
        
        except Exception as e:
            self.log_action("ERROR", "RESPONSE_EXECUTION", "FAILED", str(e))
            self.status = "FAILED"
            return {"status": "FAILED", "error": str(e)}

# ===== CONFIGURATION =====
# Update these variables with your environment details
WAZUH_API_URL = "https://wazuh-manager.example.com:55000"
WAZUH_USERNAME = "admin"
WAZUH_PASSWORD = "password"

CROWDSTRIKE_CLIENT_ID = "your_crowdstrike_client_id"
CROWDSTRIKE_CLIENT_SECRET = "your_crowdstrike_client_secret"

# ===== EXECUTION =====
# Initialize API clients
wazuh_client = WazuhAPIClient(WAZUH_API_URL, WAZUH_USERNAME, WAZUH_PASSWORD)
cs_client = CrowdStrikeForensicsClient(CROWDSTRIKE_CLIENT_ID, CROWDSTRIKE_CLIENT_SECRET)

# Execute comprehensive incident response
response = ComprehensiveWannaCryResponse(wazuh_client, cs_client)
report = response.execute_full_response()

# Print summary
print("\nINCIDENT RESPONSE SUMMARY:")
print(f"  Incident ID: {report.get('incident_id')}")
print(f"  Affected Hosts: {report.get('total_affected')}")
print(f"  Total Actions: {report.get('total_actions')}")
print(f"  Successful: {report.get('successful_actions')}")
print(f"  Failed: {report.get('failed_actions')}")
print(f"  Status: {report.get('status')}")




### 13.2. How to Use the Comprehensive Script in Jupyter Notebook

**Prerequisites:**
- Wazuh manager with API enabled
- CrowdStrike Falcon with API credentials (optional but recommended for forensics)
- Python 3.7+ with `requests` library installed

**Setup Instructions:**

1. **Install required packages:**
   ```python
   !pip install requests
   ```

2. **Copy the entire script from section 13.1 into a notebook cell** and run it to define all classes

3. **Update configuration variables** with your environment details:
   ```python
   WAZUH_API_URL = "https://your-wazuh-manager.com:55000"
   WAZUH_USERNAME = "your_username"
   WAZUH_PASSWORD = "your_password"
   
   CROWDSTRIKE_CLIENT_ID = "your_crowdstrike_client_id"
   CROWDSTRIKE_CLIENT_SECRET = "your_crowdstrike_client_secret"
   ```

4. **Execute the incident response** by running the configuration and execution section in a new cell

**Output:**
- Real-time console output showing each phase and action
- JSON incident report saved to `incident_report.json`
- Complete action log with timestamps and status for each operation
- Access results via the `report` variable for further analysis

---

**For questions or updates to this runbook, contact your Incident Response Team.**

**Document Control:**
- **Version**: 0.1
- **Classification**: Internal Use Only
- **Distribution**: Incident Response Team, Executive Leadership, IT Security
